# Analisi Phase 1: baseline Video Swin e feature tracking

Questo notebook confronta gli output di `scripts/train_phase1_baseline.py`: `baseline`, `rim_only`, `ball_only` e `all`. Cerca i `runs.csv` in `experiments/PHASE1_*`, identifica il gruppo dai metadati `run_config.json`, e ignora le cartelle di training incomplete.

La heatmap mostra la media di `val_accuracy` sui seed per fase e architettura; le celle riportano anche il numero di run. Le metriche sono di validation: il test split non viene letto.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 140
EXPERIMENTS_DIR = Path("experiments")
EXPECTED_GROUPS = ["baseline", "rim_only", "ball_only", "all"]
METRIC = "val_macro_f1"  # oppure val_accuracy


In [ ]:
def get_group(config):
    feature_config = config.get("feature_config") or config.get("experiment_config") or {}
    tracking = feature_config.get("extra", {}).get("tracking_merge")
    if tracking and tracking.get("feature_group") in EXPECTED_GROUPS:
        return tracking["feature_group"]
    if feature_config.get("tracker") == "none":
        return "baseline"
    return None

candidates = {group: [] for group in EXPECTED_GROUPS}
incomplete = []
for run_dir in EXPERIMENTS_DIR.glob("PHASE1_*"):
    config_path = run_dir / "run_config.json"
    runs_path = run_dir / "runs.csv"
    if not config_path.is_file():
        continue
    config = json.loads(config_path.read_text(encoding="utf-8"))
    group = get_group(config)
    if group is None:
        continue
    if not runs_path.is_file():
        incomplete.append(run_dir.name)
        continue
    candidates[group].append((runs_path.stat().st_mtime, run_dir, runs_path))

selected = {}
frames = []
for group, runs in candidates.items():
    if not runs:
        continue
    _, run_dir, runs_path = max(runs, key=lambda item: item[0])
    frame = pd.read_csv(runs_path)
    frame["variant"] = group
    frame["run_dir"] = str(run_dir)
    frames.append(frame)
    selected[group] = runs_path
    print(f"{group:10s} -> {runs_path}")
if incomplete:
    print("\nTraining senza runs.csv (ancora incompleti o interrotti):")
    for name in incomplete:
        print(" -", name)

if not frames:
    raise FileNotFoundError("Nessun runs.csv Phase 1 completo trovato in experiments/PHASE1_*")
df = pd.concat(frames, ignore_index=True)
df["learning_rate"] = pd.to_numeric(df["learning_rate"])
df["seed"] = pd.to_numeric(df["seed"], downcast="integer")
print("\nGruppi completi:", sorted(df.variant.unique()))
print("Metriche disponibili:", [c for c in ("val_macro_f1", "val_accuracy") if c in df.columns])

In [ ]:
# Heatmap: accuracy media in validation per fase, architettura e variante
if "val_accuracy" not in df.columns:
    raise KeyError("runs.csv non contiene la colonna val_accuracy")

acc_mean = df.pivot_table(index=["phase", "architecture"], columns="variant",
                          values="val_accuracy", aggfunc="mean")
acc_n = df.pivot_table(index=["phase", "architecture"], columns="variant",
                       values="val_accuracy", aggfunc="count").reindex_like(acc_mean)
phase_order = [p for p in ["flat", "p1", "p2", "p3"] if p in acc_mean.index.get_level_values("phase")]
acc_mean = acc_mean.reindex(pd.MultiIndex.from_product(
    [phase_order, sorted(acc_mean.index.get_level_values("architecture").unique())],
    names=["phase", "architecture"])).dropna(how="all")
acc_n = acc_n.reindex_like(acc_mean)
columns = [g for g in EXPECTED_GROUPS if g in acc_mean.columns]
acc_mean = acc_mean.reindex(columns=columns)
acc_n = acc_n.reindex(columns=columns)
labels = acc_mean.copy().astype(object)
for row in acc_mean.index:
    for col in acc_mean.columns:
        value = acc_mean.loc[row, col]
        n = acc_n.loc[row, col]
        labels.loc[row, col] = f"{value:.3f}\nn={int(n)}" if pd.notna(value) else ""

fig, ax = plt.subplots(figsize=(2.1 + 1.5 * len(columns), 2.0 + 0.55 * len(acc_mean)))
sns.heatmap(acc_mean, annot=labels, fmt="", cmap="YlGnBu", vmin=0.5, vmax=1.0,
            linewidths=0.6, linecolor="white", cbar_kws={"label": "Accuracy validation"},
            ax=ax, mask=acc_mean.isna())
ax.set_title("Accuracy media su validation per fase, architettura e feature set (5 seed)\n")
ax.set_xlabel("Gruppo di feature")
ax.set_ylabel("Fase / architettura")
ax.set_yticklabels([f"{phase} / {arch}" for phase, arch in acc_mean.index], rotation=0)
plt.tight_layout()
plt.show()
display(acc_mean.round(4).rename_axis(index=["fase", "architettura"]))


In [ ]:
# Media e deviazione standard per variante, fase e architettura
summary = (df.groupby(["variant", "phase", "architecture", "learning_rate"])[METRIC]
          .agg(mean="mean", sd="std", n="count").reset_index())
display(summary.sort_values(["phase", "architecture", "variant"]))

if df.variant.nunique() > 1:
    plot = sns.catplot(data=df, kind="bar", x="variant", y=METRIC, hue="architecture",
                       col="phase", col_wrap=2, errorbar="sd", height=4, aspect=1.2)
    plot.set_axis_labels("Feature set", METRIC)
    plot.set_titles("Fase: {col_name}")
    for ax in plot.axes.flat:
        ax.tick_params(axis="x", rotation=25)
    plt.show()
else:
    print("Per il grafico comparativo serve almeno un training completo oltre alla baseline.")

In [ ]:
# Delta appaiati rispetto alla baseline: variante - baseline, per fase/architettura/seed
KEYS = ["phase", "architecture", "learning_rate", "seed"]
baseline = df[df.variant == "baseline"]
paired_frames = []
for variant in [v for v in EXPECTED_GROUPS if v != "baseline" and v in set(df.variant)]:
    other = df[df.variant == variant]
    pair = baseline[KEYS + [METRIC]].merge(other[KEYS + [METRIC]], on=KEYS, how="outer",
                                                     suffixes=("_baseline", "_variant"), indicator=True)
    unmatched = pair[pair._merge != "both"]
    if len(unmatched):
        print(f"{variant}: confronto non appaiato, {len(unmatched)} combinazioni mancanti; verificare seed/config.")
        continue
    pair["variant"] = variant
    pair["delta"] = pair[f"{METRIC}_variant"] - pair[f"{METRIC}_baseline"]
    paired_frames.append(pair)

if paired_frames:
    paired = pd.concat(paired_frames, ignore_index=True)
    delta_summary = (paired.groupby(["variant", "phase", "architecture"])["delta"]
                    .agg(mean_delta="mean", sd_delta="std", n="count").reset_index())
    display(delta_summary.sort_values(["phase", "architecture", "variant"]))
    if paired.variant.nunique() > 1:
        sns.pointplot(data=paired, x="phase", y="delta", hue="variant", errorbar=("ci", 95), dodge=0.25)
        plt.axhline(0, color="black", linewidth=1)
        plt.title(f"Delta appaiato di {METRIC} rispetto alla baseline")
        plt.ylabel("variante - baseline")
        plt.show()
else:
    print("Nessun confronto appaiato disponibile: serve un gruppo completo oltre alla baseline.")

print("Interpretazione: delta positivo favorisce la variante. Sono metriche di validation; il test resta escluso.")